# Data pipeline and evaluation metric

The first of three notebooks. This one covers the data pipeline and the metric, the
second the models and how they are trained, the third the evaluation and the results.

Every decision behind the dataset and the metric is shown on the actual data rather than
described. Several of them were reversed during the project after inspecting the corpus
or checking the game itself, and the ones that were reversed are documented here as well.

All logic lives in `src/`; this notebook only calls it.

Run `python scripts/prepare_data.py` first.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from src.data.dedup import deduplicate
from src.data.splits import train_val_test_split
from src.data.augment import augment
from src.data.symmetry import symmetries
from src.data.vglc import extract_rooms, parse_dungeon
from src.metrics.connectivity import (access_mask, main_component_mask, preserves_topology,
                                      probe_mask, tile_accuracy, walkable_components)
from src.tiles import CHAR_MAP, IDX_TO_CHAR, RAW_COLS, RAW_ROWS, walkable_mask
from src.viz import parse_room, render_components, render_mask, render_room

RAW = "../data/raw"
plt.rcParams["figure.dpi"] = 130

## 1. The tile alphabet

Ten symbols. Traversability is not a property of the symbol alone: two of them are
context dependent, and that is the subject of section 3.

In [ ]:
for char, idx in sorted(CHAR_MAP.items(), key=lambda kv: kv[1]):
    print(f"  {idx}  {char}")
print(f"\nroom in storage: {RAW_ROWS} rows x {RAW_COLS} cols")

## 2. Storage orientation is transposed with respect to the game

The corpus stores each room as 16 rows by 11 columns, while the game presents it as 11
by 16. This was verified against the published map image of dungeon 1: a black room in
the upper middle of the file corresponds to the middle left of the map, and the
pinwheel room, the enemy room and the entire right-hand cluster line up only after
transposing.

The transpose preserves 4-adjacency, so topology is unaffected. It is a display
convention, adopted so that the figures read the way the game looks.

In [ ]:
grid = parse_dungeon(f"{RAW}/tloz1_1.txt")
raw = grid[:RAW_ROWS, :RAW_COLS]

fig, axes = plt.subplots(1, 2, figsize=(5.5, 3))
render_room(raw, axes[0], f"storage {raw.shape[0]}x{raw.shape[1]}", letters=True)
render_room(raw.T, axes[1], f"game view {raw.T.shape[0]}x{raw.T.shape[1]}", letters=True)
plt.tight_layout()

## 3. Traversability, and the two traps

`O` (element over floor) is shallow water and is walkable: treating it as an obstacle
would split rooms that are perfectly traversable.

`-` (void) is overloaded. In rooms with a black background it is the walkable floor; in
rooms that contain actual floor it is a pit that Link falls into. The rule that resolves
the ambiguity is contextual: void counts as walkable only when the room contains no `F`
tile at all.

In [ ]:
rooms, sources = extract_rooms(RAW, to_visual=True)
F = CHAR_MAP["F"]

black = [r for r in rooms if not np.any(r == F) and np.any(r == CHAR_MAP["-"])]
pit = [r for r in rooms if np.any(r == F) and np.any(r == CHAR_MAP["-"])]
print(f"black-background rooms (void is floor): {len(black)}")
print(f"rooms with floor and void   (void is a pit): {len(pit)}")

fig, axes = plt.subplots(2, 2, figsize=(6, 3.6))
render_room(black[0], axes[0][0], "black background")
render_mask(walkable_mask(black[0]), axes[0][1], "walkable: void included")
render_room(pit[0], axes[1][0], "floor plus void")
render_mask(walkable_mask(pit[0]), axes[1][1], "walkable: void excluded")
plt.tight_layout()

## 4. No content filtering

Rooms without doors were discarded twice during the project and reinstated both times.

The first filter kept only rooms with a door. Eight of the discarded rooms turned out to
contain a stair, so the filter was widened to door-or-stair. That was wrong as well:
stairs do not connect rooms of the dungeon, they lead to secret areas outside the map,
and the doorless rooms are secret rooms whose four walls are bombable. They are
traversable, and in some cases required to finish the dungeon.

The corpus does not annotate bombable walls. Filtering on their absence discards
legitimate rooms because of a limitation of the dataset, not of the rooms. The only
rooms discarded are all-void slots, which are holes in the dungeon grid rather than
rooms.

In [ ]:
D, S = CHAR_MAP["D"], CHAR_MAP["S"]
no_door = [r for r in rooms if not np.any(r == D)]
no_access = [r for r in no_door if not np.any(r == S)]

print(f"rooms extracted: {len(rooms)}")
print(f"  without a door: {len(no_door)}")
print(f"  without door or stair: {len(no_access)}")

fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.2))
for ax, r in zip(axes, no_access[:3]):
    render_room(r, ax, "sealed room")
plt.tight_layout()

## 5. Deduplication before splitting

The game reuses layouts across dungeons, sometimes mirrored. Deduplication treats two
rooms as equivalent when one is a symmetry of the other, and runs before the split so
that a room and its mirror cannot land on opposite sides of it.

In [ ]:
unique = deduplicate(rooms, mode="symmetry")
print(f"{len(rooms)} rooms -> {len(unique)} unique up to symmetry")

# external check: no pair among the survivors is a symmetry of another
keys = [min(s.tobytes() for s in symmetries(r)) for r in unique]
print(f"residual symmetry collisions: {len(keys) - len(set(keys))}")

# a template that appears several times
from collections import Counter
counts = Counter(min(s.tobytes() for s in symmetries(r)) for r in rooms)
repeated = counts.most_common(1)[0]
copies = [r for r in rooms if min(s.tobytes() for s in symmetries(r)) == repeated[0]]
print(f"most repeated layout: {repeated[1]} occurrences")

fig, axes = plt.subplots(1, min(4, len(copies)), figsize=(6.5, 1.8))
for ax, r in zip(np.atleast_1d(axes), copies[:4]):
    render_room(r, ax)
plt.tight_layout()

## 6. Split and augmentation

Three splits rather than two: validation drives hyperparameter choices, test provides
the final numbers. Using one split for both would bias the reported results.

Augmentation applies the four shape-preserving symmetries to the training split only.
The nominal gain is fourfold, but most rooms are self-symmetric, so the number of
distinct samples grows by considerably less. The augmentation is still worth applying:
the NCA is not equivariant to reflection by construction, since the Sobel filters give
each cell a sense of direction.

In [ ]:
train, val, test = train_val_test_split(unique, val_fraction=0.15, test_fraction=0.15, seed=42)
print(f"train {len(train)}   val {len(val)}   test {len(test)}")

self_sym = sum(1 for r in train if len({s.tobytes() for s in symmetries(r)}) < 4)
aug = augment(train)
distinct = len({x.tobytes() for x in aug})
print(f"\nself-symmetric training rooms: {self_sym} of {len(train)}")
print(f"augmented samples: {len(aug)}, distinct: {distinct} "
      f"({distinct / len(train):.2f}x rather than 4x)")

# no symmetry of a held-out room may appear in training
train_keys = {x.tobytes() for x in train}
leaks = sum(any(s.tobytes() in train_keys for s in symmetries(x)) for x in list(val) + list(test))
print(f"symmetry leakage into training: {leaks}")

## 7. Why the metric is relative to the pristine room

An absolute criterion would ask that every access point be reachable from the interior.
Applied to pristine rooms, it fails on a substantial number of them, because the corpus
annotates geometry but not affordances: pushable blocks, bombable walls and items that
make water traversable are simply not marked.

Reachability with movable blocks is the solution of a planning problem, not a static
property of the graph, so annotating it would not fix the issue. The criterion is
therefore defined relative to the pristine room, which is robust to this by
construction.

In [ ]:
def all_access_reachable(room):
    labels, n = walkable_components(room)
    acc = access_mask(room)
    if n == 0 or not acc.any():
        return None
    main = main_component_mask(room)
    return bool(np.all(labels[acc] == labels[main][0]))


verdicts = [all_access_reachable(r) for r in unique]
failing = [r for r, v in zip(unique, verdicts) if v is False]
print(f"pristine rooms where an access is unreachable by a naive search: {len(failing)}")

fig, axes = plt.subplots(2, 3, figsize=(6.5, 3.2))
for k, r in enumerate(failing[:3]):
    render_room(r, axes[0][k], "pristine room")
    render_components(r, axes[1][k], "components")
plt.tight_layout()

## 8. The criterion in practice

Two conditions, both necessary. The access points must be exactly those of the original,
and the probe cells must fall into the same connected components. Probes are the main
walkable component together with every access point, including accesses that lie outside
that component.

In [ ]:
room = parse_room(["WWWWW", "WFFFW", "WFFFW", "WWDWW"])

fig, axes = plt.subplots(1, 4, figsize=(7, 1.9))
render_room(room, axes[0], "room", letters=True)
render_mask(main_component_mask(room), axes[1], "main component")
render_mask(access_mask(room), axes[2], "access points")
render_mask(probe_mask(room), axes[3], "probes")
plt.tight_layout()

In [ ]:
cases = [
    ("identical copy", room.copy(), True),
    ("door walled off", None, False),
    ("door turned into floor", None, False),
    ("wall in front of the door", None, False),
    ("extra door", None, False),
    ("monster added on floor", None, True),
]
variants = [room.copy() for _ in cases]
variants[1][3, 2] = CHAR_MAP["W"]
variants[2][3, 2] = CHAR_MAP["F"]
variants[3][2, 2] = CHAR_MAP["W"]
variants[4][0, 2] = CHAR_MAP["D"]
variants[5][1, 1] = CHAR_MAP["M"]

print(f"{'case':<32}{'expected':>10}{'measured':>10}")
print("-" * 52)
for (label, _, expected), variant in zip(cases, variants):
    got = preserves_topology(room, variant)
    print(f"{label:<32}{str(expected):>10}{str(got):>10}")

## 9. The case the relative criterion exists for

This layout occurs several times in the corpus: a spiral of blocks around a central
stair. The stair is unreachable by a naive search even in the pristine room, because the
blocks are pushable and the corpus does not say so.

Under the relative criterion the pristine room passes, since the repaired room only has
to reproduce the original grouping. The converse also holds: a model that opens the wall
and connects the stair changes the grouping and fails, so it cannot game the metric by
improving the topology.

In [ ]:
spiral = parse_room(["WWWWWWW", "WFFFFFW", "WFWWWFW", "WFWSWFW",
                     "WFWWWFW", "WFFFFFW", "WWWDWWW"])
opened = spiral.copy()
opened[3, 2] = CHAR_MAP["F"]

fig, axes = plt.subplots(1, 3, figsize=(5.5, 2))
render_room(spiral, axes[0], "pristine", letters=True)
render_components(spiral, axes[1], "components")
render_components(opened, axes[2], "stair connected")
plt.tight_layout()

print(f"pristine against itself: {preserves_topology(spiral, spiral.copy())}")
print(f"stair wrongly connected: {preserves_topology(spiral, opened)}")

## 10. Per-tile accuracy is close to blind to functional failure

A single wrong tile in front of the door leaves accuracy above ninety percent and the
room without a usable exit. This is the observation the topological metric exists for,
and it is asserted in the test suite so that it is checked on every run.

In [ ]:
blocked = room.copy()
blocked[2, 2] = CHAR_MAP["W"]

fig, axes = plt.subplots(1, 2, figsize=(3.4, 1.9))
render_room(room, axes[0], "pristine")
render_room(blocked, axes[1], "one tile changed",
            highlight=(room != blocked))
plt.tight_layout()

print(f"tile accuracy:        {tile_accuracy([room], [blocked]):.3f}")
print(f"topology preserved:   {preserves_topology(room, blocked)}")